# Experiment 11: is the forecast band honest?

TiRex-2 gives 9 quantiles (10 %, 20 %, ..., 90 %). If they are honest, what happens falls below the 10 % line
in 10 % of the hours, between 10 % and 20 % in another 10 %, and so on: **every one of the 10 bins gets 10 %**
(a flat histogram; Gneiting, Balabdaoui & Raftery 2007, "PIT histogram").
- U shape (high outer bars) = the band is too narrow, the model is **overconfident**
- high left bar only = reality often colder than the whole band (forecast too warm)

No new model run: the saved forecasts of Experiments 2, 3 and 5 (temperature only).

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

RES = Path("/workspace/data/notebooks/results")
FIG = Path("/workspace/phase1-model-behavior/pair-1/experiments/figures/exp11")
FIG.mkdir(parents=True, exist_ok=True)
z2 = np.load(RES / "exp2_forecasts_50x48.npz")
z3 = np.load(RES / "exp3_forecasts_30perfront.npz")
z5 = np.load(RES / "exp5_after_front.npz")
ok = (z3["truth"][:, 6] - z3["truth"][:, 6:13].min(1)) >= 2
GROUPS = {
    "normal days (Exp 2)": [(z2["w4"], z2["truth"])],
    "front coming (start 6 h before the drop)": [(z3["A"][ok], z3["truth"][ok])],
    "just after the drop (start +6 / +12 h)": [(z5["fc +6 h"], z5["truth +6 h"]), (z5["fc +12 h"], z5["truth +12 h"])],
    "1-2 days after (start +24 / +48 h)": [(z5["fc +24 h"], z5["truth +24 h"]), (z5["fc +48 h"], z5["truth +48 h"])],
}


def pit_bins(fc, truth):
    """Bin 0 = below the 10 % quantile ... bin 9 = above the 90 % quantile, for every case and hour."""
    return (truth[:, None, :] > fc).sum(axis=1).ravel()


hist = {g: np.bincount(np.concatenate([pit_bins(f, t) for f, t in parts]), minlength=10) for g, parts in GROUPS.items()}
n = {g: h.sum() for g, h in hist.items()}
share = {g: 100 * h / h.sum() for g, h in hist.items()}
pd.DataFrame({"forecast hours": n, "below the band (%)": {g: s[0] for g, s in share.items()},
              "inside 10-90 % (%)": {g: s[1:9].sum() for g, s in share.items()},
              "above the band (%)": {g: s[9] for g, s in share.items()}}).round(1)

,forecast hours,below the band (%),inside 10-90 % (%),above the band (%)
normal days (Exp 2),57600,13.2,78.5,8.3
front coming (start 6 h before the drop),6456,50.3,45.6,4.2
just after the drop (start +6 / +12 h),12912,19.3,72.7,8.0
1-2 days after (start +24 / +48 h),12912,8.7,79.0,12.3


## Where does reality fall in the forecast distribution?

Each panel: the 10 bins between the quantiles; dashed line = 10 %, what an honest forecast gives in every bin.
Red bars = outside the 10-90 % band.

In [2]:
LABELS = ["< 10", "10-20", "20-30", "30-40", "40-50", "50-60", "60-70", "70-80", "80-90", "> 90"]
fig = make_subplots(rows=1, cols=len(GROUPS), shared_yaxes=True, horizontal_spacing=0.03,
                    subplot_titles=[f"{g}<br><sup>{n[g]:,} forecast hours</sup>" for g in GROUPS])
for c, g in enumerate(GROUPS, start=1):
    fig.add_trace(go.Bar(x=LABELS, y=share[g], marker_color=["#d62728"] + ["#6baed6"] * 8 + ["#d62728"], showlegend=False,
                         hovertemplate="%{x} %: %{y:.1f} % of the hours<extra></extra>"), row=1, col=c)
    fig.add_hline(y=10, line={"color": "white", "dash": "dash", "width": 1}, row=1, col=c)
    fig.update_xaxes(title_text="quantile bin (%)", tickangle=-45, row=1, col=c)
fig.update_yaxes(title_text="share of forecast hours (%)", row=1, col=1)
fig.update_annotations(font={"size": 12})
fig.update_layout(height=470, template="plotly_dark", margin={"t": 120},
                  title="Is the forecast band honest?<br><sup>flat at 10 % = honest; high red bars = reality outside the band "
                        "more often than the 20 % it should be</sup>")
fig.write_html(FIG / "exp11_pit_histograms.html", include_plotlyjs="cdn")
fig.show()

## Summary: below, inside, above the band

Honest = 10 % / 80 % / 10 % (dotted lines).

In [3]:
fig = go.Figure()
names = list(GROUPS)[::-1]
for lab, idx, colr in [("below the band (reality colder)", [0], "#2166ac"), ("inside the 10-90 % band", list(range(1, 9)), "#d9d9d9"),
                       ("above the band (reality warmer)", [9], "#b2182b")]:
    fig.add_trace(go.Bar(y=names, x=[share[g][idx].sum() for g in names], orientation="h", name=lab, marker_color=colr,
                         hovertemplate="%{y}: %{x:.1f} %<extra>" + lab + "</extra>"))
for x in (10, 90):
    fig.add_vline(x=x, line={"color": "white", "dash": "dot", "width": 1})
fig.update_layout(barmode="stack", height=360, template="plotly_dark", legend={"orientation": "h", "y": -0.25},
                  xaxis={"title": "share of forecast hours (%)", "range": [0, 100]},
                  title="How often is reality below, inside or above the band?")
fig.write_html(FIG / "exp11_band_summary.html", include_plotlyjs="cdn")
fig.show()